# 05 Final Load Prep

In this final notebook, we prepare the dataset for ingestion into Tableau. This involves:
1. Creating meaningful categorizations (`Severity_Label`, `Weather_Category`, `State_Region`).
2. Cleaning column names so Tableau can easily read them (removing special characters).
3. Saving the final flat file, and potentially summary tables, to be used directly in the dashboard.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import prepare_tableau_dataset

In [2]:
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded cleaned dataset: {df.shape}")

Loaded cleaned dataset: (95607, 46)


### Applying Tableau Transformations

We use the pipeline function `prepare_tableau_dataset` to apply all final mappings.

In [3]:
if TABLEAU_READY_PATH.exists():
    print(f"Loading existing Tableau-ready dataset from {TABLEAU_READY_PATH}")
    tableau_df = pd.read_csv(TABLEAU_READY_PATH)
else:
    tableau_df = prepare_tableau_dataset(df)
    TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
    tableau_df.to_csv(TABLEAU_READY_PATH, index=False)
    print(f'Saved Tableau-ready dataset to {TABLEAU_READY_PATH}')

Loading existing Tableau-ready dataset from /Users/mohitsingh/Desktop/DVA-capstone/Section-A_G12_DeliverIQ/data/processed/tableau_ready_dataset.csv


### Verification of Tableau File

In [4]:
print(f"Tableau dataset shape: {tableau_df.shape}")
print("\nNew Categorical Features:")
print("Severity Labels:", tableau_df['Severity_Label'].unique())
print("Weather Categories:", tableau_df['Weather_Category'].unique())
print("State Regions:", tableau_df['State_Region'].unique())

Tableau dataset shape: (95607, 49)

New Categorical Features:
Severity Labels: <StringArray>
['Critical', 'High', 'Moderate', 'Low']
Length: 4, dtype: str
Weather Categories: <StringArray>
['Clear', 'Cloudy', 'Rain', 'Fog', 'Snow', 'Smoke', 'Other', 'Storm', 'Wind']
Length: 9, dtype: str
State Regions: <StringArray>
['South', 'West', 'Midwest', 'Northeast']
Length: 4, dtype: str


In [5]:
tableau_df.head()

,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance_mi,Street,City,County,State,...,Hour,Is_Weekend,Time_of_Day,Season,Duration_min,Road_Feature_Count,Is_Rush_Hour,Severity_Label,Weather_Category,State_Region
0,4,2017-02-25 01:21:34,2017-02-25 07:21:34,29.949330,-95.291820,1.718,US-59 S,Humble,Harris,TX,...,1.0,1,Night,Winter,360.000000,0,0,Critical,Clear,South
1,4,2022-10-13 23:14:46,2022-10-14 01:15:36,35.037490,-89.795978,0.371,Bill Morris Pkwy,Memphis,Shelby,TN,...,23.0,0,Night,Fall,120.833333,1,0,Critical,Clear,South
2,3,2018-12-03 05:54:08,2018-12-03 06:23:59,37.824001,-122.316925,0.000,I-80 W,Oakland,Alameda,CA,...,5.0,0,Morning,Winter,29.850000,1,0,High,Cloudy,West
3,4,2019-06-24 16:18:39,2019-06-24 16:47:09,44.477478,-88.470745,0.407,W5590 State Highway 54,Black Creek,Outagamie,WI,...,16.0,0,Afternoon,Summer,28.500000,0,1,Critical,Rain,Midwest
4,2,2021-06-13 17:43:00,2021-06-13 18:46:00,33.956033,-118.109349,1.411,I-5 S,Downey,Los Angeles,CA,...,17.0,1,Evening,Summer,63.000000,1,1,Moderate,Clear,West


### Create Summary Extracts (Optional)
For faster Tableau performance, we can export pre-aggregated summary tables.

In [6]:
# State-level summary
state_summary = tableau_df.groupby(['State', 'State_Region']).agg(
    Total_Accidents=('Severity', 'count'),
    Avg_Severity=('Severity', 'mean'),
    Avg_Duration_min=('Duration_min', 'mean'),
    Pct_Rush_Hour=('Is_Rush_Hour', 'mean')
).reset_index()

state_summary['Pct_Rush_Hour'] = (state_summary['Pct_Rush_Hour'] * 100).round(2)

state_summary_path = PROJECT_ROOT / 'data/processed/tableau_state_summary.csv'
state_summary.to_csv(state_summary_path, index=False)
print(f"Saved state summary to {state_summary_path}")
state_summary.head()

Saved state summary to /Users/mohitsingh/Desktop/DVA-capstone/Section-A_G12_DeliverIQ/data/processed/tableau_state_summary.csv


,State,State_Region,Total_Accidents,Avg_Severity,Avg_Duration_min,Pct_Rush_Hour
0,AL,South,1506,2.565737,85.174978,48.27
1,AR,South,296,3.591216,114.052759,39.86
2,AZ,West,2748,2.117176,115.986845,44.43
3,CA,West,17214,2.491925,79.268133,39.01
4,CO,West,1709,3.038619,106.745212,44.82
